In [55]:
import xarray as xr
import pandas as pd
import glob
import os
import math

import numpy as np
import re

In [56]:
ppes = xr.open_dataset("/glade/work/qingyuany/repo_data/spatialtuning/simv4_iteration1.nc")
obs = xr.open_dataset("/glade/work/qingyuany/repo_data/spatialtuning/obs.nc")

In [57]:

obs_dict = {"SWCF": "swcrf_toa", "LWCF": "lwcrf_toa", "TGCLDLWP": "clwp", "TMQ": "pwv",
          "CLDTOT_ISCCP": "clt_isccp", "FLUT": "olr", "PRECT": "pr","FSNTOA": "swabs_toa"}


In [58]:

lat_bins = np.arange(-85, 86, 5)  # -90 to 90 every 10 degrees
lat_labels = ((lat_bins[:-1] + lat_bins[1:]) / 2).astype(int).astype(str)  # midpoint labels
lat_labels = np.char.add("zonal_lat_",lat_labels)


In [59]:
len(lat_labels)

34

In [60]:
ppe_zonal_list = []
obs_zonal_list = []

for cam_name, obs_name in obs_dict.items():

    ppe_da = ppes[cam_name]
    filter_tf = obs_ds[obs_name].notnull()
    
    ppe_da_filtered = ppe_da.where(filter_tf)
    
    zonal_ppe_temp = ppe_da_filtered.mean(dim  = "lon", skipna = True).groupby_bins("lat",lat_bins, labels = lat_labels).mean(dim = "lat", skipna = True).to_dataframe().unstack(level = 1)
    zonal_ppe_temp.columns.name = None
    zonal_ppe_temp.columns = ["_".join(col) for col in list(zonal_ppe_temp.columns)]

    zonal_obs_temp = obs_ds[obs_name].mean(dim = "lon", skipna = True).groupby_bins("lat",lat_bins, labels = lat_labels).mean(dim = "lat", skipna = True).to_series()
    zonal_obs_temp.index = zonal_ppe_temp.columns

    
    ppe_zonal_list.append(zonal_ppe_temp)
    obs_zonal_list.append(zonal_obs_temp)

ppe_zonal_pd = pd.concat(ppe_zonal_list, axis = 1)
obs_zonal_pd = pd.concat(obs_zonal_list)


In [61]:
#ppe_zonal_pd.to_csv("/glade/work/qingyuany/repo_data/spatialtuning/ppe_zonal_v4_iteration1.csv", index = True)
#obs_zonal_pd.to_csv("/glade/work/qingyuany/repo_data/spatialtuning/obs_zonal_v4_iteration1.csv", index = True)
